# 사내 문서 QA — v2: 답변 생성

v1에 이어 `RecursiveCharacterTextSplitter`로 분할하고, 검색된 문서 3개를 근거로 LLM이 답변을 생성한다. 문서에 없는 내용은 "문서에 없음"이라고 답하도록 시스템 프롬프트로 지시한다. (Reranker 없음)

In [ ]:
CHUNK_SIZE=500
CHUNK_OVERLAP=100

In [ ]:
query_list = [
    "연차를 사용하려면 최소 며칠 전에 신청해야 하나요?",
    "서울 출장 시 숙박비는 최대 얼마까지 지원되나요?",
    "출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?",
    "회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?",
    "재택근무를 신청하려면 어떤 조건을 충족해야 하나요?",
    "직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?",
    "출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?",
    "개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?",
    "업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?",
    "제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.",
]

In [ ]:
from dotenv import load_dotenv 
import os

load_dotenv()
os.environ["LANGSMITH_PROJECT"] = "RAG_ONE_02"

In [ ]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1} (p.{d.metadata.get('page_label')}):\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [ ]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_community.vectorstores import FAISS 
from langchain_openai import OpenAIEmbeddings 
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF 로드 
loader = PyPDFLoader("data/가상Tech_업무가이드.pdf")
documents = loader.load() 

# 문서 분할 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

split_docs = text_splitter.split_documents(documents) 

# 임베딩 
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# FAISS 벡터스토어 생성 
db = FAISS.from_documents(split_docs, embeddings)

In [ ]:
# 리트리버 생성
# Reranker 버전은 후보 7개(k=7) 중 3개를 골라 LLM에 넘겼다.
# 여기서는 Reranker가 없으므로 LLM에 넘길 문서 3개를 FAISS 검색으로 바로 가져온다.
retriever = db.as_retriever(search_kwargs={"k": 3})

In [ ]:
for i, query in enumerate(query_list):
    docs = retriever.invoke(query)
    print(f"\n질문{i+1}. {query}\n")
    pretty_print_docs(docs)

## 답변 생성

In [ ]:
from langchain_core.prompts import ChatPromptTemplate 
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    context = f"\n{'-' * 100}\n".join(
        [f"Document {i+1} (p.{d.metadata.get('page_label')}):\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    return context
    
prompt = ChatPromptTemplate.from_messages([
    ("system", """
    1. 당신은 사용자의 질문에 대답을 하는 친절한 AI agent이다.
    2. 반드시 문서에 있는 내용만으로 답한다.
    3. 사용자의 질문을 문서에서 찾을 수 없으면 문서에서 찾지 못하여 대답을 할 수 없다고 답변한다.
    4. 질문이 여러 항목이면 항목마다 문서에 있는지 판단하고, 없는 항목은 '문서에 없음'이라고 밝힌다.
    5. 답변에 근거 페이지를 밝힌다.

    문서:
    {context}"""),
    ("human", "{question}"),
])

llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0)

rag_chain = prompt | llm | StrOutputParser()

for i, query in enumerate(query_list):
    docs = retriever.invoke(query)
    context = format_docs(docs)
    answer = rag_chain.invoke({"context": context, "question": query})
    print(f"질문 {i+1}: {query}\n\n답변: {answer}\n\n{'-'*80}\n\n")